In [1]:
import os
import pickle
import jsonlines
import pandas as pd
import numpy as np
import json
import copy
from tqdm import tqdm

In [2]:
try:
    META_FILE = "../../output/trip/item2attributes.json"   # 相对 notebook 的新路径
    data = json.load(open(META_FILE, "r", encoding="utf-8"))
    print(f"Loaded metadata for {len(data)} items from {META_FILE}")
except FileNotFoundError:
    print(f"Error: {META_FILE} not found. Did grocery_data_process.py run successfully?")
    exit()
except json.JSONDecodeError:
    print(f"Error: {META_FILE} is not a valid JSON file.")
    exit()
    

Loaded metadata for 3570 items from ../../output/trip/item2attributes.json


In [3]:
example_dict = {}
for item_dict in tqdm(data.values()):
    example_dict.update(item_dict)

  0%|                                                                                                                                         | 0/3570 [00:00<?, ?it/s]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3570/3570 [00:00<00:00, 711405.61it/s]

In [4]:
# 查看数据结构
print("Sample item structure:")
sample_key = list(data.keys())[0]
print(f"Key: {sample_key}")
print(f"Value: {data[sample_key]}")

cate_dict = {}
for item_id, item_dict in tqdm(data.items()):
    if "categories" in item_dict and item_dict["categories"]:
        # TripAdvisor 的 categories 是字符串，需要分割
        categories = item_dict["categories"].split(", ")
        if len(categories) >= 1:
            cate_dict[item_id] = categories[-1]
        else:
            cate_dict[item_id] = "NA"
    else:
        cate_dict[item_id] = "NA"

Sample item structure:
Key: 113317
Value: {'name': 'Casablanca Hotel Times Square', 'type': 'hotel', 'categories': 'hotel, 4.0 star', 'address': {'region': 'NY', 'street-address': '147 West 43rd Street', 'postal-code': '10036', 'locality': 'New York City'}, 'url': 'http://www.tripadvisor.com/Hotel_Review-g60763-d113317-Reviews-Casablanca_Hotel_Times_Square-New_York_City_New_York.html', 'phone': nan, 'details': nan, 'hotel_class': 4.0, 'region_id': 60763}


  0%|                                                                                                                                         | 0/3570 [00:00<?, ?it/s]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3570/3570 [00:00<00:00, 410406.06it/s]

TripAdvisor 数据结构说明：

去掉不需要的属性（如 region_id, url），剩下的属性可以分为文本类和列表类

文本类：直接添加到prompt即可

列表类：先把列表中的element组成文本，再添加到prompt

TripAdvisor 保留字段：
- name: 酒店名称
- type: 类型（如 hotel）
- categories: 分类（字符串格式）
- address: 地址信息（字典格式）
- phone: 电话号码
- details: 详细信息
- hotel_class: 星级

舍弃字段：
- region_id: 地区ID（对推荐无意义）
- url: 链接（对嵌入质量无意义）

In [5]:
instruction = "The point of interest has the following attributes: \n "

In [6]:
FIELD_LIMITS = {
    "name": 100,
    "type": 50,
    "categories": 200,
    "address": 300,
    "details": 1000,
    "hotel_class": 20
}

DEFAULT_FIELD_LIMIT = 300


def truncate_value(value, limit):
    if value is None:
        return ""
    s = str(value)
    if len(s) <= limit:
        return s
    return s[: max(0, limit - 3)] + "..."



In [7]:
TOTAL_INPUT_CHAR_LIMIT = 4000  # 总输入字符上限，适配 4k token 级别上下文的保守值

# 字段加入优先级（高到低）。未列明字段默认最低优先级。
FIELD_PRIORITY = [
    "name",
    "type",
    "categories",
    "hotel_class",
    "address",
    "details"
]

PRIORITY_DEFAULT = 9999


In [8]:
item_data = {}
for item_id, item_dict in tqdm(data.items()):
    # 收集各字段条目，稍后基于优先级与总长度限制进行拼接
    entries = []  # (priority, entry_str)

    for key, value in item_dict.items():
        # 跳过不需要的字段
        if key in ["region_id", "url", "phone"]:
            continue

        # 生成该字段的字符串
        entry_str = None
        if key == "categories":  # 字符串形式
            if value and str(value).strip():
                val = truncate_value(str(value).replace("\n", ", "), FIELD_LIMITS.get("categories", DEFAULT_FIELD_LIMIT))
                entry_str = key + " is " + val + "; "
            else:
                entry_str = key + " is none; "
        elif key == "address":
            if isinstance(value, dict):
                address_parts = []
                for addr_key, addr_value in value.items():
                    if addr_value and str(addr_value).strip():
                        address_parts.append(f"{addr_key}: {addr_value}")
                if address_parts:
                    address_str = ", ".join(address_parts).replace("\n", ", ")
                    val = truncate_value(address_str, FIELD_LIMITS.get("address", DEFAULT_FIELD_LIMIT))
                    entry_str = key + " is " + val + "; "
                else:
                    entry_str = key + " is none; "
            else:
                val = truncate_value(str(value).replace("\n", ", "), FIELD_LIMITS.get("address", DEFAULT_FIELD_LIMIT))
                entry_str = key + " is " + val + "; "
        elif key in ["details"]:
            if value and str(value).strip():
                val = truncate_value(str(value).replace("\n", ", "), FIELD_LIMITS.get("details", DEFAULT_FIELD_LIMIT))
                entry_str = key + " is " + val + "; "
            else:
                entry_str = key + " is none; "
        else:
            if value is not None and str(value).strip():
                limit = FIELD_LIMITS.get(key, DEFAULT_FIELD_LIMIT)
                val = truncate_value(str(value).replace("\n", ", "), limit)
                entry_str = key + " is " + val + "; "
            else:
                entry_str = key + " is none; "

        # 记录条目和优先级
        priority = FIELD_PRIORITY.index(key) if key in FIELD_PRIORITY else PRIORITY_DEFAULT
        entries.append((priority, entry_str))

    # 按优先级排序并拼接，遵守总字符上限
    entries.sort(key=lambda x: x[0])
    item_prompt = copy.deepcopy(instruction)
    for _, e in entries:
        remaining = TOTAL_INPUT_CHAR_LIMIT - len(item_prompt)
        if remaining <= 0:
            break
        if len(e) <= remaining:
            item_prompt += e
        else:
            # 尝试截断该条目以适配剩余空间
            if remaining > 10:
                item_prompt += e[: max(0, remaining - 3)] + "..."
            break

    # 去掉末尾的分号与空格
    if item_prompt.endswith("; "):
        item_prompt = item_prompt[:-2]

    item_data[item_id] = item_prompt

  0%|                                                                                                                                         | 0/3570 [00:00<?, ?it/s]

 62%|████████████████████████████████████████████████████████████████████████████▍                                              | 2219/3570 [00:00<00:00, 22178.92it/s]

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3570/3570 [00:00<00:00, 21897.79it/s]

In [9]:
# 保存处理后的数据
json.dump(item_data, open("../../output/trip/item_str.json", "w"))

# 显示一个示例
print("Sample generated prompt:")
sample_key = list(item_data.keys())[0]
print(f"Item ID: {sample_key}")
print(f"Prompt: {item_data[sample_key]}")
print(f"\nTotal items processed: {len(item_data)}")

Sample generated prompt:
Item ID: 113317
Prompt: The point of interest has the following attributes: 
 name is Casablanca Hotel Times Square; type is hotel; categories is hotel, 4.0 star; hotel_class is 4.0; address is region: NY, street-address: 147 West 43rd Street, postal-code: 10036, locality: New York City; details is nan

Total items processed: 3570


In [10]:
import os

OUTPUT_DIR = "../../output/trip/"    

# convert to jsonline 
def save_data(data_path, data):
    '''write all_data list to a new jsonl'''
    outfile = os.path.join(OUTPUT_DIR, data_path)
    with jsonlines.open(outfile, "w") as w:
        for meta_data in data:
            w.write(meta_data)

# 加载 ID 映射
id_map_path = os.path.join(OUTPUT_DIR, "id_map.json")
id_map = json.load(open(id_map_path, "r"))["item2id"]

# 构建 jsonline 数据
json_data = []
for key, value in item_data.items():
    if key in id_map:  # 确保 item 在映射中存在
        json_data.append({
            "input": value, 
            "target": "", 
            "item": key, 
            "item_id": id_map[key]
        })
    else:
        print(f"Warning: Item {key} not found in id_map")

print(f"Generated {len(json_data)} jsonline entries")
save_data("item_str.jsonline", json_data)
print("✅ Successfully saved item_str.jsonline")

Generated 3570 jsonline entries
✅ Successfully saved item_str.jsonline
